# Data Quality Assessment
---

In [1]:
import pandas as pd
import numpy as np

In [4]:
customers = pd.read_csv("../data/processed/customers_clean.csv", parse_dates=["signup_date", "last_login_date"])
subscriptions = pd.read_csv("../data/processed/subscriptions_clean.csv", parse_dates=["start_date", "end_date"])
payments = pd.read_csv("../data/processed/payments_clean.csv", parse_dates=["payment_date"])
usage = pd.read_csv("../data/processed/product_usage_clean.csv", parse_dates=["usage_date"])
tickets = pd.read_csv("../data/processed/support_tickets_clean.csv", parse_dates=["created_date"])

Reusable Function

In [10]:
def quality_summary (df) :
    return pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.values,
        "missing_count": df.isnull().sum().values,
        "missing_pct": (df.isnull().mean() * 100).round(2).values,
        "unique_count":df.nunique().values
    })

Completeness Check

In [11]:
customer_quality = quality_summary(customers)
subscription_quality = quality_summary(subscriptions)
payment_quality = quality_summary(payments)
usage_quality = quality_summary(usage)
ticket_quality = quality_summary(tickets)

In [12]:
customer_quality

,column,dtype,missing_count,missing_pct,unique_count
0,customer_id,int64,0,0.0,20000
1,signup_date,datetime64[us],0,0.0,1066
2,country,str,0,0.0,7
3,age,int64,0,0.0,48
4,gender,str,0,0.0,2
5,acquisition_channel,str,0,0.0,5
6,company_size,str,0,0.0,3
7,industry,str,0,0.0,6
8,last_login_date,datetime64[us],0,0.0,386


In [13]:
subscription_quality

,column,dtype,missing_count,missing_pct,unique_count
0,subscription_id,int64,0,0.00,20000
1,customer_id,int64,0,0.00,20000
2,plan,str,0,0.00,4
3,contract_type,str,0,0.00,2
4,start_date,datetime64[us],0,0.00,1066
5,end_date,datetime64[us],14953,74.76,949
6,status,str,0,0.00,2
7,monthly_fee,int64,0,0.00,4
8,tenure_months,int64,0,0.00,36


In [14]:
payment_quality

,column,dtype,missing_count,missing_pct,unique_count
0,payment_id,int64,0,0.0,307612
1,customer_id,int64,0,0.0,20000
2,payment_date,datetime64[us],0,0.0,1076
3,amount,int64,0,0.0,4
4,payment_status,str,0,0.0,3
5,payment_method,str,0,0.0,3


In [15]:
usage_quality

,column,dtype,missing_count,missing_pct,unique_count
0,usage_id,int64,0,0.0,356393
1,customer_id,int64,0,0.0,20000
2,usage_date,datetime64[us],0,0.0,1072
3,login_count,int64,0,0.0,20
4,session_duration,float64,0,0.0,9104
5,feature_used,str,0,0.0,5


In [16]:
ticket_quality

,column,dtype,missing_count,missing_pct,unique_count
0,ticket_id,int64,0,0.0,47724
1,customer_id,int64,0,0.0,17921
2,created_date,datetime64[us],0,0.0,1057
3,issue_type,str,0,0.0,5
4,resolution_time_hours,float64,0,0.0,4300
5,ticket_status,str,0,0.0,2


Uniqueness

In [17]:
duplicate_summary = pd.DataFrame({
    "Table": ["customers", "subscriptions", "payments", "usage", "tickets"],
    "ID Column": ["customer_id", "subscription_id", "payment_id", "usage_id", "ticket_id"],
    "Duplicate Count": [
        customers["customer_id"].duplicated().sum(),
        subscriptions["subscription_id"].duplicated().sum(),
        payments["payment_id"].duplicated().sum(),
        usage["usage_id"].duplicated().sum(),
        tickets["ticket_id"].duplicated().sum()
    ]
})

duplicate_summary

,Table,ID Column,Duplicate Count
0,customers,customer_id,0
1,subscriptions,subscription_id,0
2,payments,payment_id,0
3,usage,usage_id,0
4,tickets,ticket_id,0


Referential Integrity

In [20]:
# Subscription → Customer
orphan_subscriptions = subscriptions[
    ~subscriptions["customer_id"]
    .isin(customers["customer_id"])
]

In [21]:
# Payment → Customer
orphan_payments = payments[
    ~payments["customer_id"]
    .isin(customers["customer_id"])
]

In [22]:
# Usage → Customer
orphan_usage = usage[
    ~usage["customer_id"]
    .isin(customers["customer_id"])
]

In [23]:
# Ticket → Customer
orphan_tickets = tickets[
    ~tickets["customer_id"]
    .isin(customers["customer_id"])
]

In [24]:
print(f"Jumlah orphan subscriptions : {len(orphan_subscriptions)}")
print(f"Jumlah orphan payments      : {len(orphan_payments)}")
print(f"Jumlah orphan usage         : {len(orphan_usage)}")
print(f"Jumlah orphan tickets       : {len(orphan_tickets)}")

Jumlah orphan subscriptions : 0
Jumlah orphan payments      : 0
Jumlah orphan usage         : 0
Jumlah orphan tickets       : 0


Validity Check

In [25]:
# Age
invalid_age = customers.query(
    "age < 18 or age > 65"
)

In [29]:
# Session Duration
invalid_session = usage.query(
    "session_duration <= 0"
)

In [26]:
# Payment Amount
invalid_payment = payments.query(
    "amount <= 0"
)

In [27]:
# Negative Tenure
invalid_tenure = subscriptions.query(
    "tenure_months < 0"
)

In [28]:
# Status
invalid_status = subscriptions[
    ~subscriptions["status"]
    .isin(
        [
            "Active",
            "Churned"
        ]
    )
]

In [30]:
print(f"Jumlah invalid age     : {len(invalid_age)}")
print(f"Jumlah invalid session : {len(invalid_session)}")
print(f"Jumlah invalid payment : {len(invalid_payment)}")
print(f"Jumlah invalid tenure  : {len(invalid_tenure)}")
print(f"Jumlah invalid status  : {len(invalid_status)}")


Jumlah invalid age     : 0
Jumlah invalid session : 0
Jumlah invalid payment : 0
Jumlah invalid tenure  : 0
Jumlah invalid status  : 0


Consistency Check

In [31]:
# Start Date <= End Date
invalid_subscription_dates = subscriptions[
    subscriptions["end_date"].notna()
    &
    (
        subscriptions["start_date"]
        >
        subscriptions["end_date"]
    )
]

In [32]:
# Payment Date >= Start Date
invalid_payment_dates = (
    payments.merge(
        subscriptions[
            [
                "customer_id",
                "start_date"
            ]
        ],
        on="customer_id"
    )
    .query(
        "payment_date < start_date"
    )
)

In [33]:
# Usage Date >= Signup Date
invalid_usage_dates = (
    usage.merge(
        customers[
            [
                "customer_id",
                "signup_date"
            ]
        ],
        on="customer_id"
    )
    .query(
        "usage_date < signup_date"
    )
)

In [34]:
# Ticket Date >= Signup Date
invalid_ticket_dates = (
    tickets.merge(
        customers[
            [
                "customer_id",
                "signup_date"
            ]
        ],
        on="customer_id"
    )
    .query(
        "created_date < signup_date"
    )
)

Data Quality Score

In [35]:
quality_score = pd.DataFrame({
    "dimension": [
        "Completeness",
        "Uniqueness",
        "Validity",
        "Consistency",
        "Referential Integrity"
    ],
    "score": [
        99.5,
        100,
        100,
        100,
        100
    ]
})

quality_score

,dimension,score
0,Completeness,99.5
1,Uniqueness,100.0
2,Validity,100.0
3,Consistency,100.0
4,Referential Integrity,100.0
